# Stage 3 -- Model-Ready: Daily Means

## Input
`Data/Data_Collection/Final/Stage_2/agg_market_daily_means.parquet` -- aggregated daily means table (cap-weighted mean per stock factor + macro daily factors), keyed on `date`

## Purpose
Applies expanding-window z-standardisation to the aggregated daily means table to produce the final model-ready daily means dataset. All continuous features are standardised using only historical data up to t-1. Binary, bounded, and calendar features are left in their raw form.

---

## Pipeline

### Step 1: Load
The Stage 2 aggregated daily means table is loaded and sorted by date. Shape and date range are reported.

### Step 2: Identify Columns to Z-Score vs Skip
Columns are split into two groups:

**Skipped (not z-scored):**
- `date` and `target_daily_return` -- meta columns
- **Binary/indicator features** (from Panel C macro): `is_monday`, `is_friday`, `is_quarter_end`, `is_turn_of_month`, `is_opex_week`, `vix_above_20`, `vix_above_30`, `curve_inverted_2y10y`, `curve_inverted_3m10y`, `credit_stress`. Z-scoring would destroy their interpretability and create non-standard distributions.
- **Categorical/ordinal features:** `day_of_week` (0--4), `month_of_year` (1--12), `trading_days_to_month_end` (0--22). Already on a natural bounded scale.

Only columns actually present in the data are added to the skip set. Everything else is z-scored.

### Step 3: Expanding-Window Z-Standardisation
Applied to all `zscore_cols` using the formula:

`z_t = (x_t - μ_{1:t-1}) / σ_{1:t-1}`

- **`shift(1)` applied to both expanding mean and std** -- the current observation is excluded from its own standardisation, preventing look-ahead bias
- **Minimum 252 trading days (~1 year)** before the first valid z-score is produced
- Computed in one vectorised pass using pandas `expanding().mean()` and `expanding().std()` followed by `shift(1)`
- Any resulting ±inf values (from σ = 0 periods on near-constant factors) are replaced with NaN

### Step 4: Drop Warmup Rows
The first `MIN_WINDOW + 1` rows (253 rows) are dropped. Additional rows containing any NaN in z-scored columns are then trimmed by finding the last NaN row index and slicing from the row after it -- this handles near-constant factors whose expanding variance takes longer to stabilise. The start date is then aligned to 2007-11-30 to match the combined tables. The warmup end date, total rows dropped, and final date range are reported.

### Step 5: Validate
- **NaN check:** zero NaN expected in all feature columns after warmup trim; any remaining are listed
- **Infinite value check:** confirms no ±inf remain after the replacement in Step 3
- **Zero-variance check:** identifies any columns that are all-NaN or constant after z-scoring
- **Target integrity:** mean (~0.0005), std (~0.012), min, max, NaN count -- confirms target was not z-scored
- **Z-score distribution check:** first 10 z-scored features shown with mean, std, min, max (expect mean ≈ 0, std ≈ 1)
- **Binary feature check:** confirms skipped binary columns still contain only 0/1 values
- **No duplicate dates**

### Step 6: Save
Sorted by date and saved to parquet.

---

## Key Design Decisions
- **`shift(1)` is the critical look-ahead prevention mechanism.** Without it, the current observation would be included in its own mean and std, leaking future information into early training rows.
- **`MIN_WINDOW = 252`** requires approximately one year of history before the first z-score is computed. This ensures the expanding statistics are meaningful rather than based on just a few observations.
- **Additional NaN trim after the fixed warmup drop** handles edge cases where near-constant factors (e.g., corporate event indicators that are zero for the first year) have expanding standard deviations that remain near zero beyond the 252-row window. The trim finds the last NaN row across all z-scored columns and removes everything up to and including it.
- **Start date aligned to 2007-11-30** after warmup trim to match the combined (daily + monthly) tables, enabling consistent date ranges across all Stage 3 outputs.
- **Binary and calendar features not z-scored** because they are already on interpretable bounded scales; z-scoring would obscure their meaning and produce asymmetric distributions for binary indicators.

## Output
`Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_daily_means.parquet` -- keyed on `date`, all continuous features expanding-window z-standardised using only past data, binary/calendar features in raw form, `target_daily_return` in raw returns

In [1]:
# %% [markdown]
# # Stage 3 — Model-Ready: Daily Means
#
# Applies expanding-window z-standardisation to the aggregated daily means
# table, producing the final model-ready dataset.
#
# Z-scoring approach:
#   z_t = (x_t - μ_{1:t-1}) / σ_{1:t-1}
#   - Uses ONLY data up to t-1 (shift(1) ensures no look-ahead)
#   - Minimum 252 trading days (~1 year) before first valid z-score
#   - Binary/bounded/calendar features are NOT z-scored
#   - Target variable is NOT z-scored (stays in raw returns)
#
# Input:  Stage_2/agg_market_daily_means.parquet
# Output: Stage_3_Model_Ready/model_market_daily_means.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import time

IN_PATH = Path('../../../../Data/Data_Collection/Final/Stage_2/agg_market_daily_means.parquet')
OUT_DIR = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: LOAD")
print("=" * 90)

df = pd.read_parquet(IN_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"\n  Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: IDENTIFY COLUMNS TO Z-SCORE vs SKIP
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: IDENTIFY COLUMNS TO Z-SCORE vs SKIP")
print("=" * 90)

# Meta columns — never z-scored
meta_cols = ['date', 'target_daily_return']

# Binary/bounded features — already on a natural 0/1 or small integer scale.
# Z-scoring would destroy their interpretability and create weird distributions.
skip_binary = [
    'is_monday', 'is_friday', 'is_quarter_end',
    'is_turn_of_month', 'is_opex_week',
    'vix_above_20', 'vix_above_30',
    'curve_inverted_2y10y', 'curve_inverted_3m10y',
    'credit_stress',
]

# Categorical/ordinal features — bounded, known scale
skip_categorical = [
    'day_of_week',               # 0-4 (Mon-Fri)
    'month_of_year',             # 1-12
    'trading_days_to_month_end', # 0-22
]

# Combine all skip columns (only those that actually exist in the data)
all_skip = set(meta_cols + skip_binary + skip_categorical)
all_skip = {c for c in all_skip if c in df.columns}

# Everything else gets z-scored
zscore_cols = [c for c in df.columns if c not in all_skip]

print(f"\n  Total columns: {df.shape[1]}")
print(f"  Columns to z-score: {len(zscore_cols)}")
print(f"  Columns to skip: {len(all_skip)}")

# Report what's being skipped
print(f"\n  Skipped columns:")
print(f"    Meta:        {[c for c in meta_cols if c in df.columns]}")
print(f"    Binary:      {[c for c in skip_binary if c in df.columns]}")
print(f"    Categorical: {[c for c in skip_categorical if c in df.columns]}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: EXPANDING-WINDOW Z-STANDARDISATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 3: EXPANDING-WINDOW Z-STANDARDISATION")
print("=" * 90)

MIN_WINDOW = 252  # ~1 year of trading days

t0 = time.time()

print(f"\n  Z-scoring {len(zscore_cols)} columns with expanding window (min {MIN_WINDOW} days)...")
print(f"  Formula: z_t = (x_t - μ_{{1:t-1}}) / σ_{{1:t-1}}")
print(f"  shift(1) ensures NO look-ahead — current day excluded from mean/std\n")

# Compute expanding mean and std, shifted by 1 to exclude current observation
# This is the most critical line: .shift(1) prevents look-ahead bias
expanding_mean = df[zscore_cols].expanding(min_periods=MIN_WINDOW).mean().shift(1)
expanding_std = df[zscore_cols].expanding(min_periods=MIN_WINDOW).std().shift(1)

# Apply z-score: (x - mean) / std
# Where std == 0 or NaN, result will be NaN (handled in validation)
df[zscore_cols] = (df[zscore_cols] - expanding_mean) / expanding_std

elapsed = time.time() - t0
print(f"  Z-scoring completed in {elapsed:.1f}s")


# Replace any inf/-inf from division by zero (σ = 0 for constant factors)
inf_before = np.isinf(df[zscore_cols]).sum().sum()
if inf_before > 0:
    df[zscore_cols] = df[zscore_cols].replace([np.inf, -np.inf], np.nan)
    print(f"  ⚠ Replaced {inf_before} infinite values with NaN (from σ = 0 periods)")
else:
    print(f"  ✓ No infinite values produced")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: DROP WARMUP ROWS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: DROP WARMUP ROWS")
print("=" * 90)

# First MIN_WINDOW rows have NaN z-scores (not enough history)
# Plus 1 extra row due to shift(1)
warmup_needed = MIN_WINDOW + 1

pre_drop = len(df)
warmup_date = df.iloc[warmup_needed - 1]['date']
df = df.iloc[warmup_needed:].reset_index(drop=True)


# Trim remaining NaN from near-constant factors in early expanding window
pre = len(df)
last_nan_row = max(df[df[zscore_cols].isna().any(axis=1)].index)
df = df.iloc[last_nan_row + 1:].reset_index(drop=True)
print(f"  Trimmed {pre - len(df)} additional rows to remove early z-score NaN")
print(f"  Rows: {pre:,} → {len(df):,}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")


print(f"\n  Dropped first {warmup_needed} rows (warmup period)")
print(f"  Warmup end date: {warmup_date.date()}")
print(f"  Rows: {pre_drop:,} → {len(df):,}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

# Align start date with combined tables (2007-11-30)
df = df[df['date'] >= '2007-11-30'].reset_index(drop=True)
print(f"  Aligned to combined start date: {len(df):,} rows from {df['date'].min().date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: VALIDATE")
print("=" * 90)

# 5a. Check for NaN in z-scored features
feature_cols = [c for c in df.columns if c not in ['date', 'target_daily_return']]
feature_nan = df[feature_cols].isna().sum()
feature_nan_total = feature_nan.sum()

if feature_nan_total > 0:
    nan_cols = feature_nan[feature_nan > 0].sort_values(ascending=False)
    print(f"\n  ⚠ Feature NaN: {feature_nan_total}")
    print(f"  Columns with NaN ({len(nan_cols)}):")
    for c in nan_cols.head(20).index:
        print(f"    {c}: {int(nan_cols[c])} NaN")
else:
    print(f"\n  ✓ Zero NaN in features")

# 5b. Check for infinite values (std = 0 → division by zero → inf)
inf_count = 0
inf_cols = []
for c in zscore_cols:
    n_inf = np.isinf(df[c]).sum()
    if n_inf > 0:
        inf_count += n_inf
        inf_cols.append((c, n_inf))

if inf_count > 0:
    print(f"\n  ⚠ Infinite values: {inf_count}")
    for c, n in inf_cols[:15]:
        print(f"    {c}: {n}")
else:
    print(f"  ✓ Zero infinite values")

# 5c. Check for columns with zero variance (σ = 0 throughout → all NaN after z-score)
zero_var_cols = []
for c in zscore_cols:
    if c in df.columns:
        if df[c].isna().all():
            zero_var_cols.append(c)
        elif pd.notna(df[c].std()) and float(df[c].std()) == 0:
            zero_var_cols.append(c)

if zero_var_cols:
    print(f"\n  ⚠ Zero-variance after z-score ({len(zero_var_cols)}):")
    for c in zero_var_cols:
        print(f"    {c}")
else:
    print(f"  ✓ No zero-variance columns")

# 5d. Target untouched check
print(f"\n  Target statistics (should be raw returns, NOT z-scored):")
print(f"    Mean:  {df['target_daily_return'].mean():.6f} (expect ~0.0005)")
print(f"    Std:   {df['target_daily_return'].std():.6f} (expect ~0.012)")
print(f"    Min:   {df['target_daily_return'].min():.6f}")
print(f"    Max:   {df['target_daily_return'].max():.6f}")
print(f"    NaN:   {df['target_daily_return'].isna().sum()}")

# 5e. Z-score distribution sanity check
# After z-scoring, features should have mean ≈ 0 and std ≈ 1
# (not exactly, because expanding window evolves over time)
sample_cols = [c for c in zscore_cols if c in df.columns][:10]
print(f"\n  Z-score distribution check (first 10 z-scored features):")
print(f"  {'Column':<40s} {'Mean':>8s} {'Std':>8s} {'Min':>8s} {'Max':>8s}")
print("  " + "-" * 70)
for c in sample_cols:
    vals = df[c].dropna()
    if len(vals) > 0:
        print(f"  {c:<40s} {vals.mean():>8.3f} {vals.std():>8.3f} "
              f"{vals.min():>8.2f} {vals.max():>8.2f}")

# 5f. Binary features untouched check
binary_in_data = [c for c in skip_binary if c in df.columns]
if binary_in_data:
    print(f"\n  Binary features (should still be 0/1):")
    for c in binary_in_data[:5]:
        unique = sorted(df[c].dropna().unique())
        print(f"    {c}: unique values = {unique}")

# 5g. No duplicate dates
n_dupes = df['date'].duplicated().sum()
assert n_dupes == 0, "FATAL: Duplicate dates!"
print(f"\n  ✓ No duplicate dates")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 6: SAVE")
print("=" * 90)

df = df.sort_values('date').reset_index(drop=True)

out_path = OUT_DIR / 'model_market_daily_means.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')

file_size = out_path.stat().st_size
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"    Size: {file_size / 1e6:.1f} MB")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("MODEL-READY DAILY MEANS COMPLETE")
print("=" * 90)

n_zscored = len(zscore_cols)
n_skipped = len([c for c in all_skip if c in df.columns])

print(f"""
  Input:  agg_market_daily_means.parquet (Stage 2)
  Output: model_market_daily_means.parquet (Stage 3)

  Z-standardisation:
    Method:     Expanding window, shift(1), min {MIN_WINDOW} days
    Z-scored:   {n_zscored} features
    Skipped:    {n_skipped} features (binary/calendar/target)
    Warmup:     {warmup_needed} rows dropped

  Result:
    Rows:       {df.shape[0]:,} trading days
    Columns:    {df.shape[1]}
    Dates:      {df['date'].min().date()} → {df['date'].max().date()}
    NaN:        {feature_nan_total} features + {df['target_daily_return'].isna().sum()} target

  Saved: {out_path}
""")

STEP 1: LOAD

  Loaded: 4,605 rows × 400 columns
  Date range: 2006-09-13 → 2024-12-30

STEP 2: IDENTIFY COLUMNS TO Z-SCORE vs SKIP

  Total columns: 400
  Columns to z-score: 385
  Columns to skip: 15

  Skipped columns:
    Meta:        ['date', 'target_daily_return']
    Binary:      ['is_monday', 'is_friday', 'is_quarter_end', 'is_turn_of_month', 'is_opex_week', 'vix_above_20', 'vix_above_30', 'curve_inverted_2y10y', 'curve_inverted_3m10y', 'credit_stress']
    Categorical: ['day_of_week', 'month_of_year', 'trading_days_to_month_end']

STEP 3: EXPANDING-WINDOW Z-STANDARDISATION

  Z-scoring 385 columns with expanding window (min 252 days)...
  Formula: z_t = (x_t - μ_{1:t-1}) / σ_{1:t-1}
  shift(1) ensures NO look-ahead — current day excluded from mean/std

  Z-scoring completed in 0.2s
  ⚠ Replaced 2 infinite values with NaN (from σ = 0 periods)

STEP 4: DROP WARMUP ROWS
  Trimmed 11 additional rows to remove early z-score NaN
  Rows: 4,352 → 4,341
  Date range: 2007-10-02 → 2024-